# Colab runner

Runs `src/train.py` on a Colab GPU. Setup only; all model code lives in `src/`.

**Rule:** use a GPU runtime only for training. The machine's disk is wiped when the session ends; checkpoints are saved to Google Drive.

In [1]:
!nvidia-smi # the !  means run this as a terminal command on the Colab machine

Sun Nov  1 12:18:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In later sessions, if the repo is already cloned, use !git pull instead

In [3]:
!git clone https://github.com/yagmur-cam/mars-terrain-segmentation.git
%cd mars-terrain-segmentation
!ls


Cloning into 'mars-terrain-segmentation'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 139 (delta 74), reused 86 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 3.03 MiB | 23.01 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/mars-terrain-segmentation
LICENSE  notebooks  README.md  requirements.txt  results  src


#### Getting the data

In [4]:
!df -h /content # check free disk space on Colab machine

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   71G  38% /


In [5]:
  # copy the dataset from Drive to the fast local disk, then unzip

!cp /content/drive/MyDrive/ncam.7z /content/
!7z x -y -o/content/data /content/ncam.7z > /dev/null
!find /content/data -maxdepth 3 -type d

/content/data
/content/data/ncam
/content/data/ncam/images
/content/data/ncam/images/edr
/content/data/ncam/images/mxy
/content/data/ncam/images/rng-30m
/content/data/ncam/labels
/content/data/ncam/labels/test
/content/data/ncam/labels/train


In [7]:
!git pull

Already up to date.


In [11]:
!git checkout private
!git pull
!grep num_workers src/train.py

Already on 'private'
Your branch is up to date with 'origin/private'.
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 6), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.84 KiB | 628.00 KiB/s, done.
From https://github.com/yagmur-cam/mars-terrain-segmentation
   13be219..664a388  private    -> origin/private
Updating 13be219..664a388
Fast-forward
 notebooks/colab_runner.ipynb | 88 ++++++++++++++++++++------------------------
 src/train.py                 | 28 +++++++++-----
 2 files changed, 58 insertions(+), 58 deletions(-)
    p.add_argument("--num_workers", type=int, default=0)
    train_loader = DataLoader(dataset=train_data, batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
    val_loader = DataLoader(dataset=val_data, batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers)


In [12]:
!grep autocast src/train.py

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):


In [13]:
# timing test

!python -m src.train --data_dir /content/data/ncam --out_dir /content/drive/MyDrive/mars_runs/test4 --size 256 --epochs 1 --limit 2000 --num_workers 2

Epoch number: 0, Elapsed time: 58s, Train loss: 0.8714, Validation loss: 0.8198, mIoU: 0.3670, IoU: [0.523, 0.595, 0.349, 0.0]
Saved checkpoint to /content/drive/MyDrive/mars_runs/test4/best.pth
Saved checkpoint to /content/drive/MyDrive/mars_runs/test4/last.pth


In [ ]:
# start training

!python -m src.train --data_dir /content/data/ncam --out_dir /content/drive/MyDrive/mars_runs/run1_256 --size 256 --epochs 20 --num_workers 2